# 05 — PMI Word Associations

This notebook uses Pointwise Mutual Information (PMI) to identify word associations.

Where TF-IDF asks "which lexemes are distinctive for a genre?", PMI asks "which lexemes tend to appear near each other?"

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/ALPcourse_Biblical_Hebrew_Project/biblical_hebrew_genre_analysis")
NOTEBOOK_DIR = PROJECT_DIR / "notebooks"
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_DIR / "output"
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"
DOCS_DIR = PROJECT_DIR / "docs"
POSTER_DIR = PROJECT_DIR / "poster"

for folder in [PROCESSED_DIR, TABLES_DIR, FIGURES_DIR, DOCS_DIR, POSTER_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

os.chdir(NOTEBOOK_DIR)
print("Project directory:", PROJECT_DIR)
print("Current folder:", os.getcwd())


!pip -q install pandas matplotlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from itertools import combinations
import math

## 2. Load content tokens

In [ ]:
content = pd.read_csv(PROCESSED_DIR / "biblical_hebrew_content_tokens.csv")
content["lex_clean"] = content["lex_clean"].astype(str).str.strip()

# Sort tokens by textual order to preserve local context.
content = content.sort_values(["book", "chapter", "verse", "word_node"]).reset_index(drop=True)

# Readable labels for figures. The analysis still uses BHSA lexeme codes.
gloss_lookup = (
    content.dropna(subset=["lex_clean", "gloss"])
    .assign(lex_clean=lambda df: df["lex_clean"].astype(str).str.strip(),
            gloss=lambda df: df["gloss"].astype(str).str.strip())
    .drop_duplicates("lex_clean")
    .set_index("lex_clean")["gloss"]
    .to_dict()
)

def display_label(lex):
    lex = str(lex).strip()
    gloss = gloss_lookup.get(lex, lex)
    return f"{gloss} ({lex})"

print("Content tokens:", len(content))
print(content.groupby("genre").size())

content.head()


## 3. Build co-occurrence counts

PMI is calculated within genre. Two lexemes are treated as co-occurring if they appear in the same verse. This is a transparent choice for a pilot project because verses are meaningful textual units and easier to explain than arbitrary window sizes.

In [ ]:
MIN_WORD_COUNT = 5
MIN_PAIR_COUNT = 2

all_pmi_rows = []

for genre, genre_df in content.groupby("genre"):
    verse_groups = genre_df.groupby(["book", "chapter", "verse"])["lex_clean"].apply(list)

    word_counts = Counter()
    pair_counts = Counter()
    total_windows = 0

    for words in verse_groups:
        # Use unique lexemes per verse so repeated words do not dominate co-occurrence.
        unique_words = sorted(set(w for w in words if isinstance(w, str) and w.strip()))
        if len(unique_words) < 2:
            continue

        total_windows += 1
        word_counts.update(unique_words)
        pair_counts.update(combinations(unique_words, 2))

    for (w1, w2), pair_count in pair_counts.items():
        if pair_count < MIN_PAIR_COUNT:
            continue
        if word_counts[w1] < MIN_WORD_COUNT or word_counts[w2] < MIN_WORD_COUNT:
            continue

        p_pair = pair_count / total_windows
        p_w1 = word_counts[w1] / total_windows
        p_w2 = word_counts[w2] / total_windows

        pmi = math.log2(p_pair / (p_w1 * p_w2))

        all_pmi_rows.append({
            "genre": genre,
            "word1": w1,
            "word2": w2,
            "word1_display": display_label(w1),
            "word2_display": display_label(w2),
            "pair_display": f"{display_label(w1)}  +  {display_label(w2)}",
            "pair_count": pair_count,
            "word1_count": word_counts[w1],
            "word2_count": word_counts[w2],
            "pmi": pmi,
        })

pmi_df = pd.DataFrame(all_pmi_rows).sort_values(["genre", "pmi"], ascending=[True, False])
pmi_df.to_csv(TABLES_DIR / "pmi_word_associations.csv", index=False)

top_pmi_by_genre = pmi_df.groupby("genre").head(25).reset_index(drop=True)
top_pmi_by_genre.to_csv(TABLES_DIR / "top_pmi_by_genre.csv", index=False)

top_pmi_by_genre.head(50)


## 4. PMI summary by genre

In [ ]:
summary = (
    top_pmi_by_genre
    .groupby("genre")["pair_display"]
    .apply(lambda pairs: "; ".join(list(pairs)[:10]))
    .reset_index(name="top_pmi_pairs")
)

summary.to_csv(TABLES_DIR / "pmi_summary_by_genre.csv", index=False)
summary


## 5. Visualize top PMI pairs

In [ ]:
# Remove old, less readable figure versions from earlier drafts.
for old in FIGURES_DIR.glob("top_pmi_*.png"):
    if not old.name.endswith("_friendly.png"):
        old.unlink()

for genre, group in top_pmi_by_genre.groupby("genre"):
    plot_data = group.head(12).copy().sort_values("pmi", ascending=True)

    plt.figure(figsize=(10, 7))
    plt.barh(plot_data["pair_display"], plot_data["pmi"])
    plt.xlabel("PMI score")
    plt.ylabel("Lexeme pair: gloss and BHSA code")
    plt.title(f"Top lexical associations in {genre} — PMI")
    plt.tight_layout()

    filename = FIGURES_DIR / f"top_pmi_{genre}_friendly.png"
    plt.savefig(filename, dpi=200, bbox_inches="tight")
    plt.show()

    print("Saved:", filename)


## 6. Interpretation checkpoint

PMI should be interpreted carefully. It highlights unusually strong associations, not necessarily the most important words overall. Minimum count thresholds are used to reduce noise from rare pairs.